In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rohanharode07/webmd-drug-reviews-dataset")

print("Path to dataset files:", path)

/Users/hungnguyen/study - master IT 2024/Term 4/research/research-code/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/hungnguyen/.cache/kagglehub/datasets/rohanharode07/webmd-drug-reviews-dataset/versions/1


In [2]:
# List the head of dataset files
import os
print("Files in dataset directory:", os.listdir(path))
# Load the dataset using pandas
import pandas as pd
dataset_file = os.path.join(path, "webmd.csv")
df = pd.read_csv(dataset_file)
df.head()

Files in dataset directory: ['webmd.csv']


,Age,Condition,Date,Drug,DrugId,EaseofUse,Effectiveness,Reviews,Satisfaction,Sex,Sides,UsefulCount
0,75 or over,Stuffy Nose,9/21/2014,25dph-7.5peh,146724,5,5,I'm a retired physician and of all the meds I ...,5,Male,"Drowsiness, dizziness , dry mouth /nose/thro...",0
1,25-34,Cold Symptoms,1/13/2011,25dph-7.5peh,146724,5,5,cleared me right up even with my throat hurtin...,5,Female,"Drowsiness, dizziness , dry mouth /nose/thro...",1
2,65-74,Other,7/16/2012,warfarin (bulk) 100 % powder,144731,2,3,why did my PTINR go from a normal of 2.5 to ov...,3,Female,,0
3,75 or over,Other,9/23/2010,warfarin (bulk) 100 % powder,144731,2,2,FALLING AND DON'T REALISE IT,1,Female,,0
4,35-44,Other,1/6/2009,warfarin (bulk) 100 % powder,144731,1,1,My grandfather was prescribed this medication ...,1,Male,,1


In [3]:
# get 1000 rows from df 
# df = df.sample(10000)
# print to the new file called webmd_1000.csv
# df.to_csv("webmd_1000.csv", index=False)


# df = df[["Age", "Condition", "Drug", "DrugId", "Satisfaction", "Sex", "Reviews"]]

df.isnull().sum()

Age               0
Condition         0
Date              0
Drug              0
DrugId            0
EaseofUse         0
Effectiveness     0
Reviews          43
Satisfaction      0
Sex               0
Sides             0
UsefulCount       0
dtype: int64

In [4]:
df = df.dropna()

for col in df.columns:
    if df[col].dtype.kind == "O":
        df[col] = df[col].str.strip()


def satisfaction_to_class(rating: int) -> int:
    """Map WebMD 1–5 satisfaction to 0=Negative, 1=Neutral, 2=Positive."""
    r = int(rating)
    if r <= 2:
        return 0
    if r == 3:
        return 1
    return 2

def filter_anomalies(df: pd.DataFrame, min_review_len: int = 10) -> pd.DataFrame:
    """Remove nulls and very short/uninformative reviews."""
    before = len(df)
    df = df.dropna(subset=["Reviews"]).copy()
    df = df[df["Reviews"].str.strip().str.len() >= min_review_len].copy()
    print(f"\n[3] Anomaly filter: removed {before - len(df)} rows "
          f"(null or < {min_review_len} chars).  Remaining: {len(df):,}")
    return df
 

In [5]:
# Negation words — preserve even after stopword removal
NEGATIONS = {
    "no", "not", "nor", "never", "neither", "hardly", "barely",
    "scarcely", "without", "cannot", "didn't", "doesn't", "wasn't",
    "shouldn't", "wouldn't", "couldn't", "won't", "can't", "don't",
}
 
CONTRACTIONS = {
    r"n't":   " not",
    r"'re":   " are",
    r"'s":    " is",
    r"'d":    " would",
    r"'ll":   " will",
    r"'ve":   " have",
    r"'m":    " am",
    r"won't": "will not",
    r"can't": "cannot",
}

import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer
    nltk.download("stopwords", quiet=True)
    nltk.download("punkt", quiet=True)
    STOPWORDS = set(stopwords.words("english"))
    STEMMER = PorterStemmer()
    HAS_NLTK = True
except ImportError:
    # Fallback: compact built-in stopword list
    STOPWORDS = {
        "i", "me", "my", "we", "our", "you", "your", "he", "she", "it", "its", "they", "them",
        "what", "which", "who", "this", "that", "these", "those", "am", "is", "are", "was",
        "were", "be", "been", "being", "have", "has", "had", "do", "does", "did", "doing",
        "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "of",
        "at", "by", "for", "with", "about", "against", "between", "into", "through", "during",
        "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on",
        "off", "over", "under", "then", "once", "here", "there", "when", "where", "why", "how",
        "all", "each", "few", "more", "most", "other", "some", "such", "than", "too", "very",
        "just", "can", "will", "should", "now", "s", "t", "re", "ve", "ll", "d", "m",
    }
    STEMMER = None
    HAS_NLTK = False

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    VADER = SentimentIntensityAnalyzer()
    HAS_VADER = True
except ImportError:
    VADER = None
    HAS_VADER = False

try:
    from imblearn.over_sampling import SMOTE  # noqa: F401
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False

In [6]:
import unicodedata
import re

def clean_text(text: str,
               remove_stopwords: bool = True,
               stem: bool = False) -> str:
    """
    Full text cleaning pipeline:
      - Lowercase
      - Normalise unicode (accents -> ASCII)
      - Remove HTML tags
      - Expand common contractions
      - Remove URLs and emails
      - Remove punctuation
      - Remove standalone digits
      - Tokenise on whitespace
      - Remove stopwords (preserve negations)
      - Optional Porter stemming
      - Drop single-character tokens
    """
    if not isinstance(text, str):
        return ""
 
    text = text.lower()
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"<[^>]+>", " ", text)                    # strip HTML
 
    for pattern, replacement in CONTRACTIONS.items():
        text = re.sub(pattern, replacement, text)
 
    text = re.sub(r"http\S+|www\.\S+|\S+@\S+", " ", text)   # URLs / emails
    text = re.sub(r"[^\w\s]", " ", text)                    # punctuation
    text = re.sub(r"\b\d+\b", " ", text)                    # standalone digits
 
    tokens = text.split()
 
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS or t in NEGATIONS]
 
    if stem and STEMMER is not None:
        tokens = [STEMMER.stem(t) for t in tokens]
 
    tokens = [t for t in tokens if len(t) > 1 or t == "i"]
 
    return " ".join(tokens)

## Machine learning pipeline

The cells below map WebMD 1–5 satisfaction to three classes (negative / neutral / positive), clean review text, hold out a test set, then train and evaluate **Multinomial NB**, **Random Forest + TF-IDF**, and an **LSTM**. Helpers are documented in code; long-running cells are marked.

In [7]:
import re

import altair as alt
import numpy as np
from IPython.display import display
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

def confusion_matrix_altair(y_true, y_pred, labels=None):
    """Confusion matrix as an Altair heatmap (counts in cells)."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    if labels is None:
        labels = sorted(np.unique(np.concatenate([y_true.ravel(), y_pred.ravel()])).tolist())
    rows = [
        {"True": str(labels[i]), "Predicted": str(labels[j]), "Count": int(cm[i, j])}
        for i in range(len(labels))
        for j in range(len(labels))
    ]
    src = pd.DataFrame(rows)
    base = alt.Chart(src).encode(
        x=alt.X("Predicted:O", title="Predicted"),
        y=alt.Y("True:O", title="True"),
    )
    heat = base.mark_rect().encode(
        color=alt.Color("Count:Q", scale=alt.Scale(scheme="blues")),
        tooltip=["True", "Predicted", "Count"],
    ).properties(width=320, height=280, title="Confusion matrix")
    text = base.mark_text(baseline="middle", fontSize=11).encode(text="Count:Q")
    display(heat + text)


# Modeling frame: encoded labels + cleaned text (tokenizer/vectorizer fit only on train later).
mldf = df.copy()
mldf["Satisfaction"] = mldf["Satisfaction"].astype(int).map(satisfaction_to_class)
mldf["Reviews"] = mldf["Reviews"].apply(clean_text)
mldf = filter_anomalies(mldf)

train_index, test_index = train_test_split(
    mldf.index, test_size=0.25, random_state=0, shuffle=True
)
train_set = mldf.loc[train_index]
test_set = mldf.loc[test_index]
print("train_set:", train_set.shape, "test_set:", test_set.shape)


[3] Anomaly filter: removed 46020 rows (null or < 10 chars).  Remaining: 316,743
train_set: (237557, 12) test_set: (79186, 12)


### Reusable trainers and evaluation

The next cell defines `report_classification`, `train_multinomial_nb`, `train_random_forest_tfidf`, and LSTM helpers. Sklearn models use **sparse** matrices (no `.toarray()`). The LSTM uses `LabelEncoder` and `sparse_categorical_crossentropy` so class indices stay consistent with `classification_report`.

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Embedding, LSTM, SpatialDropout1D
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer


def report_classification(
    model,
    X_train,
    X_test,
    y_train,
    y_test,
    *,
    target_names=None,
    plot_confusion=True,
):
    """Train/test accuracy, macro F1, classification report, optional confusion heatmap."""
    y_pred_tr = model.predict(X_train)
    y_pred_te = model.predict(X_test)
    print(f"\nAccuracy (train): {accuracy_score(y_train, y_pred_tr):.4f}")
    print(f"Accuracy (test):  {accuracy_score(y_test, y_pred_te):.4f}")
    print(f"F1 macro (test):  {f1_score(y_test, y_pred_te, average='macro'):.4f}\n")
    print(classification_report(y_test, y_pred_te, target_names=target_names, digits=2))
    if plot_confusion:
        confusion_matrix_altair(y_test, y_pred_te)


def train_multinomial_nb(
    train_df,
    test_df,
    *,
    drop_neutral=False,
    max_features=2500,
    min_df=10,
    max_df=0.8,
    alpha=1.0,
):
    """Bag-of-words (sparse) + Multinomial Naive Bayes. Optionally drop Satisfaction == 1 (neutral)."""
    train_df = train_df.copy()
    test_df = test_df.copy()
    if drop_neutral:
        train_df = train_df[train_df["Satisfaction"] != 1]
        test_df = test_df[test_df["Satisfaction"] != 1]
    vectorizer = CountVectorizer(max_features=max_features, min_df=min_df, max_df=max_df)
    X_train = vectorizer.fit_transform(train_df["Reviews"])
    X_test = vectorizer.transform(test_df["Reviews"])
    y_train = train_df["Satisfaction"].to_numpy()
    y_test = test_df["Satisfaction"].to_numpy()
    model = MultinomialNB(alpha=alpha)
    model.fit(X_train, y_train)
    classes = sorted(pd.unique(np.concatenate([y_train, y_test])))
    target_names = [str(c) for c in classes]
    report_classification(model, X_train, X_test, y_train, y_test, target_names=target_names)
    return model, vectorizer


def train_random_forest_tfidf(
    train_df,
    test_df,
    *,
    drop_neutral=False,
    max_features=2500,
    min_df=10,
    max_df=0.8,
    min_samples_split=6,
    random_state=0,
    n_jobs=-1,
    n_estimators=100,
):
    """TF-IDF (sparse) + RandomForest. Lower n_estimators for faster runs during development."""
    train_df = train_df.copy()
    test_df = test_df.copy()
    if drop_neutral:
        train_df = train_df[train_df["Satisfaction"] != 1]
        test_df = test_df[test_df["Satisfaction"] != 1]
    vectorizer = TfidfVectorizer(max_features=max_features, min_df=min_df, max_df=max_df)
    X_train = vectorizer.fit_transform(train_df["Reviews"])
    X_test = vectorizer.transform(test_df["Reviews"])
    y_train = train_df["Satisfaction"].to_numpy()
    y_test = test_df["Satisfaction"].to_numpy()
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        min_samples_split=min_samples_split,
        random_state=random_state,
        n_jobs=n_jobs,
    )
    model.fit(X_train, y_train)
    classes = sorted(pd.unique(np.concatenate([y_train, y_test])))
    target_names = [str(c) for c in classes]
    report_classification(model, X_train, X_test, y_train, y_test, target_names=target_names)
    return model, vectorizer


def build_lstm_binary(num_words=2500, maxlen=200, embedding_dim=100, spatial_dropout=0.2):
    """Embedding -> SpatialDropout1D -> LSTM -> Dense softmax for 2 classes (encoded 0/1)."""
    model = Sequential(
        [
            Embedding(num_words, embedding_dim, input_length=maxlen),
            SpatialDropout1D(spatial_dropout),
            LSTM(embedding_dim),
            Dense(2, activation="softmax"),
        ]
    )
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def train_lstm_sentiment(
    train_reviews,
    test_reviews,
    y_train_raw,
    y_test_raw,
    *,
    num_words=2500,
    maxlen=200,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
):
    """
    Tokenizer fit only on training reviews. y_* must be original labels {0, 2} after dropping neutral.
    Returns model, tokenizer, label_encoder, history.
    """
    le = LabelEncoder()
    y_train = le.fit_transform(y_train_raw)
    y_test = le.transform(y_test_raw)
    tokenizer = Tokenizer(num_words=num_words, split=" ", lower=False)
    tokenizer.fit_on_texts(train_reviews)
    X_train = pad_sequences(tokenizer.texts_to_sequences(train_reviews), maxlen=maxlen)
    X_test = pad_sequences(tokenizer.texts_to_sequences(test_reviews), maxlen=maxlen)
    model = build_lstm_binary(num_words=num_words, maxlen=maxlen)
    history = model.fit(
        X_train,
        y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=validation_split,
        callbacks=[
            EarlyStopping(
                monitor="val_loss",
                patience=2,
                min_delta=1e-4,
                restore_best_weights=True,
            )
        ],
        verbose=1,
    )
    return model, tokenizer, le, history, X_train, X_test, y_train, y_test


def evaluate_keras_binary(model, X_test, y_test_int, *, batch_size=64):
    """Test loss/accuracy and sklearn report (encoded labels 0/1 with names Negative/Positive)."""
    loss, acc = model.evaluate(X_test, y_test_int, batch_size=batch_size, verbose=0)
    print(f"\nLoss (test): {loss:.4f}\nAccuracy (test): {acc:.4f}\n")
    proba = model.predict(X_test, batch_size=batch_size, verbose=0)
    pred = np.argmax(proba, axis=-1)
    target_names = ["Negative", "Positive"]
    print(classification_report(y_test_int, pred, target_names=target_names, digits=2))
    confusion_matrix_altair(y_test_int, pred, labels=[0, 1])


def predict_sentiment(text, model, tokenizer, maxlen, label_encoder):
    """clean_review -> sequence -> pad -> predict -> 'Negative' or 'Positive' (WebMD codes 0 / 2)."""
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=maxlen)
    idx = int(np.argmax(model.predict(padded, verbose=0), axis=-1))
    original = int(label_encoder.inverse_transform([idx])[0])
    return "Positive" if original == 2 else "Negative"

### Multinomial Naive Bayes + `CountVectorizer`

Two runs: **3-class** (all satisfaction codes) and **binary** (drop neutral `Satisfaction == 1`).

In [9]:
%%time
nb_3class, _ = train_multinomial_nb(train_set, test_set, drop_neutral=False)


Accuracy (train): 0.6644
Accuracy (test):  0.6622
F1 macro (test):  0.5540

              precision    recall  f1-score   support

           0       0.69      0.70      0.70     31953
           1       0.30      0.19      0.23     10682
           2       0.70      0.77      0.73     36551

    accuracy                           0.66     79186
   macro avg       0.56      0.55      0.55     79186
weighted avg       0.64      0.66      0.65     79186



alt.LayerChart(...)

CPU times: user 4.05 s, sys: 155 ms, total: 4.2 s
Wall time: 4.38 s


In [10]:
%%time
nb_binary, _ = train_multinomial_nb(train_set, test_set, drop_neutral=True)


Accuracy (train): 0.7807
Accuracy (test):  0.7794
F1 macro (test):  0.7776

              precision    recall  f1-score   support

           0       0.78      0.74      0.76     31953
           2       0.78      0.81      0.80     36551

    accuracy                           0.78     68504
   macro avg       0.78      0.78      0.78     68504
weighted avg       0.78      0.78      0.78     68504



alt.LayerChart(...)

CPU times: user 3.53 s, sys: 120 ms, total: 3.65 s
Wall time: 3.87 s


### Random Forest + `TfidfVectorizer`

**Note:** Full `n_estimators=100` on ~220k×2500 sparse features can take **tens of minutes**. For a quicker check, rerun `train_random_forest_tfidf(..., n_estimators=25)` in a scratch cell or change the default in the helper cell above.

In [11]:
%%time
rf_3class, _ = train_random_forest_tfidf(train_set, test_set, drop_neutral=False)


Accuracy (train): 0.9934
Accuracy (test):  0.7798
F1 macro (test):  0.7012

              precision    recall  f1-score   support

           0       0.77      0.83      0.80     31953
           1       0.93      0.33      0.49     10682
           2       0.77      0.87      0.82     36551

    accuracy                           0.78     79186
   macro avg       0.83      0.68      0.70     79186
weighted avg       0.79      0.78      0.77     79186



alt.LayerChart(...)

CPU times: user 18min 20s, sys: 14.6 s, total: 18min 35s
Wall time: 3min


In [12]:
%%time
rf_binary, _ = train_random_forest_tfidf(train_set, test_set, drop_neutral=True)


Accuracy (train): 0.9964
Accuracy (test):  0.8535
F1 macro (test):  0.8525

              precision    recall  f1-score   support

           0       0.85      0.83      0.84     31953
           2       0.85      0.88      0.86     36551

    accuracy                           0.85     68504
   macro avg       0.85      0.85      0.85     68504
weighted avg       0.85      0.85      0.85     68504



alt.LayerChart(...)

CPU times: user 10min 51s, sys: 8.58 s, total: 11min
Wall time: 1min 49s


### LSTM (binary: negative vs positive)

Tokenizer is fit on **training** reviews only. `maxlen=200` and `num_words=2500` match the vectorizer caps above for a fair-ish comparison.

In [13]:
%%time
LSTM_MAXLEN = 200
tr_bin = train_set[train_set["Satisfaction"] != 1]
te_bin = test_set[test_set["Satisfaction"] != 1]
lstm_model, lstm_tokenizer, le_lstm, lstm_history, _, X_test_lstm, _, y_test_lstm = train_lstm_sentiment(
    tr_bin["Reviews"].to_numpy(),
    te_bin["Reviews"].to_numpy(),
    tr_bin["Satisfaction"].to_numpy(),
    te_bin["Satisfaction"].to_numpy(),
    maxlen=LSTM_MAXLEN,
)

Epoch 1/15


/Users/hungnguyen/study - master IT 2024/Term 4/research/research-code/venv/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


2890/2890 ━━━━━━━━━━━━━━━━━━━━ 345s 119ms/step - accuracy: 0.8045 - loss: 0.4211 - val_accuracy: 0.8368 - val_loss: 0.3666
Epoch 2/15
2890/2890 ━━━━━━━━━━━━━━━━━━━━ 341s 118ms/step - accuracy: 0.8508 - loss: 0.3391 - val_accuracy: 0.8529 - val_loss: 0.3349
Epoch 3/15
2890/2890 ━━━━━━━━━━━━━━━━━━━━ 341s 118ms/step - accuracy: 0.8680 - loss: 0.3054 - val_accuracy: 0.8610 - val_loss: 0.3212
Epoch 4/15
2890/2890 ━━━━━━━━━━━━━━━━━━━━ 345s 119ms/step - accuracy: 0.8805 - loss: 0.2811 - val_accuracy: 0.8604 - val_loss: 0.3184
Epoch 5/15
2890/2890 ━━━━━━━━━━━━━━━━━━━━ 326s 113ms/step - accuracy: 0.8907 - loss: 0.2601 - val_accuracy: 0.8670 - val_loss: 0.3167
Epoch 6/15
2890/2890 ━━━━━━━━━━━━━━━━━━━━ 328s 113ms/step - accuracy: 0.8989 - loss: 0.2418 - val_accuracy: 0.8679 - val_loss: 0.3259
Epoch 7/15
2890/2890 ━━━━━━━━━━━━━━━━━━━━ 336s 116ms/step - accuracy: 0.9083 - loss: 0.2232 - val_accuracy: 0.8691 - val_loss: 0.3221
CPU times: user 57min 3s, sys: 6min 22s, total: 1h 3min 25s
Wall time: 39

In [14]:
evaluate_keras_binary(lstm_model, X_test_lstm, y_test_lstm)


Loss (test): 0.3144
Accuracy (test): 0.8672

              precision    recall  f1-score   support

    Negative       0.86      0.86      0.86     31953
    Positive       0.87      0.88      0.88     36551

    accuracy                           0.87     68504
   macro avg       0.87      0.87      0.87     68504
weighted avg       0.87      0.87      0.87     68504



alt.LayerChart(...)

In [15]:
predict_sentiment(
    "The drug is expensive but it is worth every cent.",
    lstm_model,
    lstm_tokenizer,
    LSTM_MAXLEN,
    le_lstm,
)

/var/folders/y0/nrw5b2_123j9snqryfz7gfm80000gn/T/ipykernel_76559/678291125.py:178: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  idx = int(np.argmax(model.predict(padded, verbose=0), axis=-1))


'Positive'